# Train/test split

Before any decision is made based on the data, we set aside 20% of it as a **test set**, which will be used only once, at the very end, to estimate the performance of the final model (see [Training and test sets](../notes/train_test_split.ipynb) and [Data leakage](../notes/data_leakage.ipynb)).

Since `median_income` is the strongest predictor and the dataset is modest in size, we use **stratified sampling** on income categories, so that both sets have the same income distribution (see [Stratified sampling](../notes/stratified_sampling.ipynb) for the comparison with a purely random split and for the alternative of stratifying by the binned target).

In [ ]:
import pandas as pd
import numpy as np

housing = pd.read_csv("../data/housing.csv")

## Income categories

`median_income` is continuous, so we discretize it into 5 categories with `pd.cut`: blocks of 1.5 up to 6, and a last category grouping the whole right tail.

In [ ]:
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],  # bin edges
                               labels=[1, 2, 3, 4, 5])
housing["income_cat"].value_counts().sort_index()

income_cat
1     822
2    6581
3    7236
4    3639
5    2362
Name: count, dtype: int64

## Stratified split

In [ ]:
from sklearn.model_selection import train_test_split

strat_train_set, strat_test_set = train_test_split(
    housing, test_size=0.2, stratify=housing["income_cat"], random_state=42)

The proportion of each income category is the same in the test set as in the whole dataset:

In [ ]:
pd.DataFrame({
    "Overall %": housing["income_cat"].value_counts(normalize=True) * 100,
    "Test set %": strat_test_set["income_cat"].value_counts(normalize=True) * 100,
}).sort_index().round(2)

,Overall %,Test set %
income_cat,,
1,3.98,4.00
2,31.88,31.88
3,35.06,35.05
4,17.63,17.64
5,11.44,11.43


The `income_cat` column was only a help for stratifying, so we remove it from both sets.

In [ ]:
strat_train_set = strat_train_set.drop(columns="income_cat")
strat_test_set = strat_test_set.drop(columns="income_cat")

## Separating features and labels

Finally, we separate the **features** (`X`) from the **target** (`y`) in each set, which is how scikit-learn models expect them.

In [ ]:
X_train = strat_train_set.drop("median_house_value", axis=1)
y_train = strat_train_set["median_house_value"].copy()

X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((16512, 9), (4128, 9), (16512,), (4128,))

## Data loading utility

The split above is consolidated into a reusable function in [`utils/load_california.py`](utils/load_california.py), which loads the CSV, creates the income categories, performs the stratified split, removes the helper column and separates features and target. Every subsequent notebook loads the data through it, so all of them work with exactly the same split:

In [ ]:
from utils.load_california import load_housing_data

X_train, X_test, y_train, y_test = load_housing_data()
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((16512, 9), (4128, 9), (16512,), (4128,))

From here on, the test set is not looked at again until [`07_hyperparameter_tuning.ipynb`](07_hyperparameter_tuning.ipynb).